In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

In [2]:
# Read Data
# =========================================================
# 1) PATHS
# =========================================================
data_path = Path("../data/processed/yrbs_recoded.csv")
tables_dir = Path("../outputs/tables")
figures_dir = Path("../outputs/figures")

tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

# =========================================================
# 2) LOAD DATA
# =========================================================
df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print(df.head())

Dataset loaded successfully.
Shape: (14041, 114)
   RaceEth  HowOldAreYou  WhatIsYourSex  InWhatGradeAreYou  \
0      7.0           4.0            2.0                2.0   
1      5.0           7.0            2.0                2.0   
2      NaN           NaN            2.0                NaN   
3      7.0           1.0            1.0                1.0   
4      7.0           1.0            1.0                5.0   

   AreYouHispanicOrLatino WhatIsYourRace  HowTallAreYouWithoutShoesInMeters  \
0                     1.0              C                                NaN   
1                     2.0              E                               1.70   
2                     NaN            NaN                                NaN   
3                     1.0              A                               1.63   
4                     1.0              B                                NaN   

   HowMuchDoYouWeighWithoutShoesInKG  BicyleHelmetUse  SeatBeltUse  ...  \
0                           

# PART 1: ONE-SAMPLE Proportion Inference
#### Variable: CurrentCigaretteUse


## Research Question (Proportion Analysis)
#### Is the proportion of students who currently use cigarettes different from 0.20?

##### Behavior Variable: CurrentCigaretteUse_binary
##### 1 = success = currently uses cigarettes
##### 0 = failure = does not currently use cigarettes
##### Benchmark proportion p0 = 0.20

In [3]:
# define sample size, number of successes and proportion
behavior = df["CurrentCigaretteUse_binary"].dropna()

n_behavior = len(behavior)
x_success = (behavior == 1).sum()
phat = x_success / n_behavior
p0 = 0.20

print("Sample size (n):", n_behavior)
print("Number of successes (x):", x_success)
print("Sample proportion (p-hat):", phat)

Sample size (n): 13323
Number of successes (x): 2589
Sample proportion (p-hat): 0.1943256023418149


In [4]:
# checking if normal approximation is reasonable
np0 = n_behavior * p0
nq0 = n_behavior * (1 - p0)

print("Condition check:")
print("n * p0 =", np0)
print("n * (1-p0) =", nq0)

if np0 >= 10 and nq0 >= 10:
    print("Condition satisfied: Normal approximation is reasonable.")
else:
    print("Condition NOT satisfied: Interpret with caution.")

Condition check:
n * p0 = 2664.6000000000004
n * (1-p0) = 10658.400000000001
Condition satisfied: Normal approximation is reasonable.


In [5]:
# calculating 95% confidence interval
confidence = 0.95
alpha = 1 - confidence
z_crit = stats.norm.ppf(1 - alpha/2)

se_phat = np.sqrt(phat * (1 - phat) / n_behavior)
me_prop = z_crit * se_phat

ci_prop_lower = phat - me_prop
ci_prop_upper = phat + me_prop

print("95% Confidence Interval for population proportion:")
print((ci_prop_lower, ci_prop_upper))

95% Confidence Interval for population proportion:
(np.float64(0.18760679992328702), np.float64(0.20104440476034277))


In [6]:
# one-sample proportion z test
se_null = np.sqrt(p0 * (1 - p0) / n_behavior)

z_stat = (phat - p0) / se_null
p_value_prop = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print("One-Sample Proportion z-Test Results")
print("z-statistic =", z_stat)
print("p-value =", p_value_prop)

One-Sample Proportion z-Test Results
z-statistic = -1.637422637406872
p-value = 0.10154219139871867


In [7]:
alpha = 0.05

print("Hypotheses:")
print("H0: p = 0.20")
print("Ha: p != 0.20")

if p_value_prop < alpha:
    prop_decision = "Reject H0"
    prop_conclusion = "There is sufficient evidence that the population proportion of students who currently use cigarettes is different from 0.20."
else:
    prop_decision = "Fail to reject H0"
    prop_conclusion = "There is not sufficient evidence that the population proportion of students who currently use cigarettes is different from 0.20."

print("Decision:", prop_decision)
print("Conclusion:", prop_conclusion)

Hypotheses:
H0: p = 0.20
Ha: p != 0.20
Decision: Fail to reject H0
Conclusion: There is not sufficient evidence that the population proportion of students who currently use cigarettes is different from 0.20.


## Interpretation:
##### The estimated proportion of students who currently use cigarettes is 0.1943.
##### The 95% confidence interval is (0.1876, 0.2010).
##### The test statistic is z = -1.637, with p-value = 0.1015.
##### There is not sufficient evidence that the population proportion of students who currently use cigarettes is different from 0.20.

# PART 2: ONE-SAMPLE MEAN INFERENCE
#### Variable: BMIPCT

## Research Question (Mean Analysis):
#### Is the mean BMIPCT of students different from 65?


##### Continuous Variable: BMIPCT
##### Meaning: Body Mass Index Percentile
##### Benchmark mean mu0 = 65.0

In [8]:
# Prepare Data
bmi = pd.to_numeric(df["BMIPCT"], errors="coerce")
bmi = bmi.dropna()
bmi = bmi[(bmi >= 0) & (bmi <= 100)]

n_bmi = len(bmi)
mean_bmi = bmi.mean()
std_bmi = bmi.std(ddof=1)
mu0 = 65.0

print("Sample size (n):", n_bmi)
print("Sample mean:", mean_bmi)
print("Sample standard deviation:", std_bmi)

Sample size (n): 13062
Sample mean: 64.82068317411306
Sample standard deviation: 27.516756256180642


In [9]:
# 95% Confidence Interval for Population Mean
confidence = 0.95
alpha = 1 - confidence

se_bmi = std_bmi / np.sqrt(n_bmi)
t_crit = stats.t.ppf(1 - alpha/2, df=n_bmi - 1)
me_bmi = t_crit * se_bmi

ci_mean_lower = mean_bmi - me_bmi
ci_mean_upper = mean_bmi + me_bmi

print("95% Confidence Interval for population mean:")
print((ci_mean_lower, ci_mean_upper))

95% Confidence Interval for population mean:
(np.float64(64.34874975076964), np.float64(65.29261659745647))


In [10]:
# One-Sample t-Test
t_stat, p_value_mean = stats.ttest_1samp(bmi, popmean=mu0)

print("One-Sample t-Test Results")
print("t-statistic =", t_stat)
print("p-value =", p_value_mean)

One-Sample t-Test Results
t-statistic = -0.7447810972514642
p-value = 0.45641746445843656


In [11]:
# Decision and Interpretation
alpha = 0.05

print("Hypotheses:")
print("H0: mu = 65")
print("Ha: mu != 65")

if p_value_mean < alpha:
    mean_decision = "Reject H0"
    mean_conclusion = "There is sufficient evidence that the population mean BMIPCT is different from 65."
else:
    mean_decision = "Fail to reject H0"
    mean_conclusion = "There is not sufficient evidence that the population mean BMIPCT is different from 65."

print("Decision:", mean_decision)
print("Conclusion:", mean_conclusion)

Hypotheses:
H0: mu = 65
Ha: mu != 65
Decision: Fail to reject H0
Conclusion: There is not sufficient evidence that the population mean BMIPCT is different from 65.


## Interpretation:
##### The estimated mean BMIPCT is 64.8207.
##### The 95% confidence interval is (64.3487, 65.2926).
##### The test statistic is t = -0.745, with p-value = 0.4564.
##### There is not sufficient evidence that the population mean BMIPCT is different from 65.

# PART 3: Summary Table of Main Inferential Results

In [12]:
# Create Summary Table
inference_summary = pd.DataFrame({
    "Analysis Type": ["Proportion", "Mean"],
    "Variable": ["CurrentCigaretteUse", "BMIPCT"],
    "Sample Size": [n_behavior, n_bmi],
    "Estimate": [phat, mean_bmi],
    "Benchmark": [p0, mu0],
    "CI Lower": [ci_prop_lower, ci_mean_lower],
    "CI Upper": [ci_prop_upper, ci_mean_upper],
    "Test Statistic": [z_stat, t_stat],
    "p-value": [p_value_prop, p_value_mean],
    "Decision": [prop_decision, mean_decision]
})

print("=== Summary Table of Main Inferential Results ===")
print(inference_summary)

=== Summary Table of Main Inferential Results ===
  Analysis Type             Variable  Sample Size   Estimate  Benchmark  \
0    Proportion  CurrentCigaretteUse        13323   0.194326        0.2   
1          Mean               BMIPCT        13062  64.820683       65.0   

    CI Lower   CI Upper  Test Statistic   p-value           Decision  
0   0.187607   0.201044       -1.637423  0.101542  Fail to reject H0  
1  64.348750  65.292617       -0.744781  0.456417  Fail to reject H0  


In [13]:
# Saveing Summary Table
inference_summary.to_csv(tables_dir / "inference_summary_table.csv", index=False)
print("Saved: outputs/tables/inference_summary_table.csv")

Saved: outputs/tables/inference_summary_table.csv


# PART 4: Final Synthesis

In [14]:
print("=== FINAL SYNTHESIS ===")

print("\n[Behavior Variable: CurrentCigaretteUse]")
print("Research Question: Is the proportion of students who currently use cigarettes different from 0.20?")
print(f"Sample proportion = {phat:.4f}")
print(f"95% CI = ({ci_prop_lower:.4f}, {ci_prop_upper:.4f})")
print(f"z = {z_stat:.3f}, p-value = {p_value_prop:.4f}")
print(f"Decision: {prop_decision}")
print(prop_conclusion)

print("\n[Continuous Variable: BMIPCT]")
print("Research Question: Is the mean BMIPCT of students different from 65?")
print(f"Sample mean = {mean_bmi:.4f}")
print(f"95% CI = ({ci_mean_lower:.4f}, {ci_mean_upper:.4f})")
print(f"t = {t_stat:.3f}, p-value = {p_value_mean:.4f}")
print(f"Decision: {mean_decision}")
print(mean_conclusion)

=== FINAL SYNTHESIS ===

[Behavior Variable: CurrentCigaretteUse]
Research Question: Is the proportion of students who currently use cigarettes different from 0.20?
Sample proportion = 0.1943
95% CI = (0.1876, 0.2010)
z = -1.637, p-value = 0.1015
Decision: Fail to reject H0
There is not sufficient evidence that the population proportion of students who currently use cigarettes is different from 0.20.

[Continuous Variable: BMIPCT]
Research Question: Is the mean BMIPCT of students different from 65?
Sample mean = 64.8207
95% CI = (64.3487, 65.2926)
t = -0.745, p-value = 0.4564
Decision: Fail to reject H0
There is not sufficient evidence that the population mean BMIPCT is different from 65.


In [15]:
# =========================================================
# PART 5: SAVE FINAL PROJECT SUMMARY
# =========================================================
summary_dir = Path("../outputs/summary")
summary_dir.mkdir(parents=True, exist_ok=True)

final_summary_path = summary_dir / "final_summary.txt"

final_summary_text = f"""
PROJECT CYCLE 2 - FINAL SUMMARY
================================

Dataset:
YRBS_2007.csv

Behavior Variable for Proportion Analysis:
CurrentCigaretteUse

Continuous Variable for Mean Analysis:
BMIPCT

------------------------------------------------------------
1. RESEARCH QUESTIONS
------------------------------------------------------------

Behavior Variable:
Is the proportion of students who currently use cigarettes different from 0.20?

Continuous Variable:
Is the mean BMIPCT of students different from 65.0?

------------------------------------------------------------
2. DATA AND VARIABLES
------------------------------------------------------------

Behavior Variable:
- Variable used: CurrentCigaretteUse_binary
- Success = currently uses cigarettes
- Failure = does not currently use cigarettes
- Benchmark proportion: p0 = 0.20

Continuous Variable:
- Variable used: BMIPCT
- Valid range used: 0 to 100
- Benchmark mean: mu0 = 65.0

------------------------------------------------------------
3. EDA HIGHLIGHTS
------------------------------------------------------------

Behavior Variable:
- Final usable sample size: {n_behavior}
- Estimated proportion of current cigarette users: {phat:.4f}

Continuous Variable:
- Final usable sample size: {n_bmi}
- Mean BMIPCT: {mean_bmi:.4f}
- Standard deviation of BMIPCT: {std_bmi:.4f}

------------------------------------------------------------
4. INFERENTIAL RESULTS
------------------------------------------------------------

A. Proportion Analysis: CurrentCigaretteUse

- Sample proportion (p-hat): {phat:.4f}
- 95% Confidence Interval: ({ci_prop_lower:.4f}, {ci_prop_upper:.4f})
- Test statistic (z): {z_stat:.4f}
- p-value: {p_value_prop:.6f}
- Decision: {prop_decision}

Interpretation:
{prop_conclusion}

B. Mean Analysis: BMIPCT

- Sample mean (x-bar): {mean_bmi:.4f}
- 95% Confidence Interval: ({ci_mean_lower:.4f}, {ci_mean_upper:.4f})
- Test statistic (t): {t_stat:.4f}
- p-value: {p_value_mean:.6f}
- Decision: {mean_decision}

Interpretation:
{mean_conclusion}

------------------------------------------------------------
5. FINAL CONCLUSION
------------------------------------------------------------

This project examined one behavior variable and one continuous variable from the YRBS 2007 dataset.

For the behavior variable, the analysis estimated the proportion of students who currently use cigarettes and tested whether this proportion differs from 0.20.

For the continuous variable, the analysis estimated the mean BMIPCT and tested whether it differs from 65.0.

The confidence intervals and hypothesis tests provide statistical evidence to support the final conclusions above. These inferential results should be interpreted together with the EDA findings and with awareness of possible limitations such as missing data or unusual observations.

============================================================
End of Summary
============================================================
"""

# Save to txt file
with open(final_summary_path, "w", encoding="utf-8") as f:
    f.write(final_summary_text)

print(f"Saved: {final_summary_path}")
print("\n=== FINAL SUMMARY PREVIEW ===\n")
print(final_summary_text)

Saved: ..\outputs\summary\final_summary.txt

=== FINAL SUMMARY PREVIEW ===


PROJECT CYCLE 2 - FINAL SUMMARY

Dataset:
YRBS_2007.csv

Behavior Variable for Proportion Analysis:
CurrentCigaretteUse

Continuous Variable for Mean Analysis:
BMIPCT

------------------------------------------------------------
1. RESEARCH QUESTIONS
------------------------------------------------------------

Behavior Variable:
Is the proportion of students who currently use cigarettes different from 0.20?

Continuous Variable:
Is the mean BMIPCT of students different from 65.0?

------------------------------------------------------------
2. DATA AND VARIABLES
------------------------------------------------------------

Behavior Variable:
- Variable used: CurrentCigaretteUse_binary
- Success = currently uses cigarettes
- Failure = does not currently use cigarettes
- Benchmark proportion: p0 = 0.20

Continuous Variable:
- Variable used: BMIPCT
- Valid range used: 0 to 100
- Benchmark mean: mu0 = 65.0

-----

# PART 6: Interpretation Block

## 1. Proportion Analysis
##### The estimated proportion of students who currently use cigarettes was 0.1943.
##### A 95% confidence interval for the population proportion was (0.1876, 0.2010).
##### The one-sample z-test gave z = -1.637 with p-value = 0.1015.
##### Therefore, we fail to reject h0 and conclude that the population proportion of students who currently use cigarettes is not significantly different from 0.20.

## 2. Mean Analysis
##### The estimated mean BMIPCT was 64.8207.
##### A 95% confidence interval for the population mean was (64.3487, 65.2926).
##### The one-sample t-test gave t = -0.745 with p-value = 0.4564.
##### Therefore, we fail to reject h0 and conclude that the population mean BMIPCT is not significantly different from 65.